In [1]:
from imblearn.over_sampling import SMOTE
from pandas import DataFrame
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline
from churn.loader import DataLoader
import pandas as pd
import numpy as np
import os
import joblib
import mlflow

In [2]:
df: DataFrame = DataLoader(data_path="../data/processed/dataset_clean.csv").load()

df.head(10)

,gender,SeniorCitizen,MultipleLines,Partner,Dependents,tenure,PhoneService,OnlineSecurity,OnlineBackup,DeviceProtection,...,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,InternetService_DSL,InternetService_Fiber optic,InternetService_No,Contract_Month-to-month,Contract_One year,Contract_Two year
0,1,0,0,1,0,1,0,0,1,0,...,0,0,1,0,1,0,0,1,0,0
1,0,0,0,0,0,34,1,1,0,1,...,0,0,0,1,1,0,0,0,1,0
2,0,0,0,0,0,2,1,1,1,0,...,0,0,0,1,1,0,0,1,0,0
3,0,0,0,0,0,45,0,1,0,1,...,1,0,0,0,1,0,0,0,1,0
4,1,0,0,0,0,2,1,0,0,0,...,0,0,1,0,0,1,0,1,0,0
5,1,0,1,0,0,8,1,0,0,1,...,0,0,1,0,0,1,0,1,0,0
6,0,0,1,0,1,22,1,0,1,0,...,0,1,0,0,0,1,0,1,0,0
7,1,0,0,0,0,10,0,1,0,0,...,0,0,0,1,1,0,0,1,0,0
8,1,0,1,1,0,28,1,0,0,1,...,0,0,1,0,0,1,0,1,0,0
9,0,0,0,0,1,62,1,1,1,0,...,1,0,0,0,1,0,0,0,1,0


In [3]:
mlflow.set_experiment("churn_experimentos")
mlflow.sklearn.autolog(log_datasets=False)

X = df.drop(columns=["target",  "TotalCharges"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Treino: {len(X_train)} clientes | Teste: {len(X_test)} clientes")
print(f"Prevalência de churn — treino: {y_train.mean():.1%} | teste: {y_test.mean():.1%}")

Treino: 5634 clientes | Teste: 1409 clientes
Prevalência de churn — treino: 26.5% | teste: 26.5%


In [4]:

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    # ("SMOTE", SMOTE(random_state=42)),
    ("logistic", LogisticRegression(random_state=42)),
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

2026/10/22 13:49:27 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '6255574dd3aa4306b88327e220ead0a5', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/10/22 13:49:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/22 13:49:29 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
/home/eberte-sampaio/PycharmProjects/tech-challenge-fase-1/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:3424: FutureWarning: `y_pred` wa

              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.64      0.55      0.59       374

    accuracy                           0.80      1409
   macro avg       0.74      0.72      0.73      1409
weighted avg       0.79      0.80      0.79      1409



In [5]:
cv = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
metricas = ["recall", "precision", "f1", "average_precision", "roc_auc"]


In [ ]:
modelos = {
    'logistic': Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    'logistic_balanced': Pipeline([
        ('scaler', StandardScaler()),
        ("modelo", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")),
    ]),
    'logistic_smote': Pipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42)),
        ("modelo", LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    'random_forest': Pipeline([
        ("modelo", RandomForestClassifier(random_state=42, n_estimators=300, class_weight="balanced")),
    ]),
    'gradient_boosting': Pipeline([
        ("modelo", HistGradientBoostingClassifier(class_weight="balanced", random_state=42)),
    ]),
}

linha = []

for nome, modelo in modelos.items():
    r = cross_validate(modelo, X_train, y_train, cv=cv, scoring=metricas)
    linha.append({"modelo": nome, **{metrica: r[f"test_{metrica}"].mean() for metrica in metricas}})

comparacao = pd.DataFrame(linha).set_index("modelo").round(3)
comparacao.sort_values("average_precision", ascending=False)

In [ ]:
# for nome, dados in {"com TotalCharges": X_train,
#                     "sem TotalCharges": X_train.drop(columns="TotalCharges")}.items():
#     r = cross_validate(modelos["logistic_balanced"], dados, y_train, cv=cv, scoring="average_precision")
#     print(f"{nome}: {r['test_score'].mean():.3f}")

In [8]:
from sklearn.model_selection import GridSearchCV

grade = {"modelo__C": [0.001, 0.01, 0.1, 1, 10, 100]}

busca_log = GridSearchCV(modelos["logistic_balanced"], grade, cv=cv,
                         scoring="average_precision", return_train_score=True)
busca_log.fit(X_train, y_train)

print(busca_log.best_params_, f"{busca_log.best_score_:.3f}")
pd.DataFrame(busca_log.cv_results_)[["param_modelo__C", "mean_train_score", "mean_test_score"]]

2026/10/22 13:50:40 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '939ba58ff1ff4726abc849dc56494aad', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/10/22 13:50:41 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/eberte-sampaio/PycharmProjects/tech-challenge-fase-1/.venv/lib/python3.14/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing value

{'modelo__C': 1} 0.658


,param_modelo__C,mean_train_score,mean_test_score
0,0.001,0.643687,0.642260
1,0.010,0.658435,0.655064
2,0.100,0.662071,0.658088
3,1.000,0.662606,0.658278
4,10.000,0.662156,0.657451
5,100.000,0.661855,0.657002


In [9]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve

melhor = busca_log.best_estimator_

proba_cv = cross_val_predict(melhor, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
precisao, recall, limiares = precision_recall_curve(y_train, proba_cv)

f1 = 2 * precisao * recall / (precisao + recall + 1e-12)
i = np.argmax(f1[:-1])
limiar = limiares[i]

print(f"Limiar: {limiar:.2f} | precisão: {precisao[i]:.3f} | recall: {recall[i]:.3f}")

Limiar: 0.56 | precisão: 0.551 | recall: 0.755


In [10]:
from sklearn.metrics import classification_report, average_precision_score

proba_teste = melhor.predict_proba(X_test)[:, 1]
y_pred = (proba_teste >= limiar).astype(int)

print(classification_report(y_test, y_pred, target_names=["Não saiu", "Saiu"]))
print(f"PR-AUC no teste: {average_precision_score(y_test, proba_teste):.3f}")

              precision    recall  f1-score   support

    Não saiu       0.89      0.76      0.82      1035
        Saiu       0.52      0.73      0.61       374

    accuracy                           0.75      1409
   macro avg       0.71      0.75      0.71      1409
weighted avg       0.79      0.75      0.76      1409

PR-AUC no teste: 0.627


In [11]:
coeficientes = pd.Series(melhor.named_steps["modelo"].coef_[0], index=X_train.columns).sort_values()

coeficientes

MonthlyCharges                            -0.861344
tenure                                    -0.753943
InternetService_No                        -0.724633
Contract_Two year                         -0.339334
OnlineSecurity                            -0.111138
Dependents                                -0.110144
InternetService_DSL                       -0.101355
TechSupport                               -0.078910
PaymentMethod_Bank transfer (automatic)   -0.059101
PaymentMethod_Credit card (automatic)     -0.058339
Contract_One year                         -0.036317
PaymentMethod_Mailed check                -0.021883
gender                                    -0.019824
Partner                                    0.007927
OnlineBackup                               0.013371
PhoneService                               0.047914
SeniorCitizen                              0.055451
DeviceProtection                           0.066067
PaymentMethod_Electronic check             0.122144
PaperlessBil

In [12]:
modelo_final = busca_log.best_estimator_
modelo_final.fit(X, y)

os.makedirs("../models", exist_ok=True)
joblib.dump({
    "modelo": modelo_final,
    "limiar": float(limiar),
    "colunas": list(X.columns),
}, "../models/churn_model.joblib")

2026/10/22 13:50:46 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'cc1a151e025848a6b204bd045f3d08a9', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/10/22 13:50:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/22 13:50:48 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
/home/eberte-sampaio/PycharmProjects/tech-challenge-fase-1/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:3424: FutureWarning: `y_pred` wa

['../models/churn_model.joblib']